# 🏨 Playground · Del crudo al dataset final (preparación de datos)

⚠️ Este notebook es **autónomo**: **no** importa nada de `src/` ni de `ml_hotel_cancellations`. Es el **puente** entre la exploración (`01_eda_exploracion.ipynb`) y el modelado (`03_modelos_supervisados.ipynb`).

> En el EDA *descubrimos* qué pasaba con los datos. Aquí **aplicamos cada hallazgo** y construimos el **dataset listo para modelar**: limpiamos filas, quitamos columnas con fuga, creamos las *features* `has_company`/`has_agent`, **reducimos la cardinalidad** de `agent`/`country`/`company`, codificamos *one-hot* y guardamos el resultado en `data/processed/`.

> 🔑 **Idea metodológica central:** todo lo que el preprocesado *aprende* de los datos (qué categorías conservar, el vocabulario *one-hot*, valores de imputación…) se ajusta **solo con `train`** y se aplica a `test`. Si lo ajustáramos con todo el dataset, estaríamos **mirando el futuro** y nuestras métricas serían **optimistas**. Lo explicamos a fondo más abajo.

## 0. Configuración

Importamos librerías (incluida `train_test_split` de *scikit-learn*) y fijamos la **semilla** para que todo sea **reproducible** (`RANDOM_STATE = 42`).

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from sklearn.model_selection import train_test_split

pio.renderers.default = "notebook_connected"
import warnings; warnings.filterwarnings("ignore")

RANDOM_STATE = 42
COLOR_HOTEL = {"City Hotel": "#d95f02", "Resort Hotel": "#2c7fb8"}

## 1. Carga del crudo y hoja de ruta

Cargamos el dataset **tal cual** (igual que en el §1 del EDA) y traducimos cada **hallazgo del EDA** a una **acción** concreta de preparación.

| Hallazgo (EDA) | Acción aquí |
|---|---|
| `reservation_status` / `reservation_status_date` revelan el desenlace | **Quitar** (fuga directa) |
| `required_car_parking_spaces` se asigna en *check-in* (fuga sutil, §11) | **Quitar** |
| `assigned_room_type` se asigna en *check-in* (fuga, §12.8) | **Quitar** (`reserved_room_type` sí se conserva) |
| `arrival_date_year`: años parciales, no generaliza | **Quitar** (la estación la dan `month`/`week`) |
| ~180 reservas sin huéspedes y ~715 sin noches | **Eliminar filas** sin sentido |
| `adr` con valores extremos (−6,4 y 5400) | **Recortar** esos *outliers* |
| `children` con 4 nulos | **Imputar** a 0 |
| Nulo de `company` = *sin empresa*; nulo de `agent` = *sin agencia* (§5) | `has_company`, `has_agent`; `company` NA → `no_company` |
| `agent`/`country`/`company` con cientos de categorías (§13) | **Reducir cardinalidad** (señal extrema + soporte; resto → `Otros`) |
| Categóricas acotadas con señal fuerte (`deposit_type`…) | **One-hot** |
| ~34 % de filas son **duplicados** reales (§9) | **Mantener**, pero documentar la fuga por duplicación |

> Todas estas decisiones salieron del EDA. Aquí **no** volvemos a justificarlas: las **ejecutamos**.

In [2]:
PATH_RAW = "../../data/raw/dataset_practica_final.csv"
df = pd.read_csv(PATH_RAW, na_values=["NULL", "NA", "NaN", ""])
print("Crudo:", df.shape)
df.head(3)

Crudo: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02


## 2. Limpieza segura (fila a fila)

Empezamos por las transformaciones **seguras**: las que se deciden **fila a fila** y **no aprenden ningún parámetro** del conjunto. Quitar columnas, imputar `children` a 0, derivar features booleanas y eliminar filas absurdas **no** puede causar fuga train/test, así que podemos hacerlo **antes** de partir.

> Las transformaciones que *sí* aprenden (qué categorías conservar, el vocabulario *one-hot*…) las dejamos para **después** de la partición.

### 2.1 Quitar columnas con fuga o que no generalizan

In [3]:
COLUMNAS_FUERA = [
    "reservation_status",          # fuga directa: revela el desenlace
    "reservation_status_date",     # fuga directa
    "required_car_parking_spaces", # fuga sutil: se asigna en el check-in
    "assigned_room_type",          # fuga: se conoce en el check-in (§12.8); reserved_room_type sí se conserva
    "arrival_date_year",           # años parciales, no generaliza
]
df = df.drop(columns=COLUMNAS_FUERA)
print("Tras quitar columnas:", df.shape)

Tras quitar columnas: (119390, 27)


### 2.2 Imputar `children` (4 nulos) y derivar *features*

In [4]:
# children: 4 nulos -> 0 (no hay niños)
df["children"] = df["children"].fillna(0).astype(int)

# Features de "ausencia informativa" (§5 del EDA): el nulo es señal, no ruido
df["has_company"] = df["company"].notna().astype(int)
df["has_agent"] = df["agent"].notna().astype(int)

# Total de noches (usada también para detectar reservas sin estancia)
df["noches"] = df["stays_in_week_nights"] + df["stays_in_weekend_nights"]

df[["company", "has_company", "agent", "has_agent", "noches"]].head(3)

,company,has_company,agent,has_agent,noches
0,NaN,0,NaN,0,0
1,NaN,0,NaN,0,0
2,NaN,0,NaN,0,1


### 2.3 Eliminar filas sin sentido y recortar `adr`

In [5]:
n0 = len(df)

# Reservas sin ningun huesped (adults + children + babies == 0)
sin_huespedes = (df["adults"] + df["children"] + df["babies"]) == 0
# Reservas con 0 noches de estancia
sin_noches = df["noches"] == 0
# adr extremos: un negativo (-6.4) y un atipico enorme (5400)
adr_extremo = (df["adr"] < 0) | (df["adr"] >= 5400)

quitar = sin_huespedes | sin_noches | adr_extremo
df = df[~quitar].reset_index(drop=True)

print(f"Filas sin huespedes : {sin_huespedes.sum():>6}")
print(f"Filas sin noches    : {sin_noches.sum():>6}")
print(f"Filas con adr extremo: {adr_extremo.sum():>5}")
print(f"\nTotal eliminadas    : {n0 - len(df):>6}  ({(n0-len(df))/n0:.1%})")
print(f"Filas restantes     : {len(df):>6}")

Filas sin huespedes :    180
Filas sin noches    :    715
Filas con adr extremo:     2

Total eliminadas    :    827  (0.7%)
Filas restantes     : 118563


## 3. Partición train/test (¡antes de ajustar nada!)

Separamos `X` (predictoras) e `y` (`is_canceled`) y hacemos una partición **estratificada** (mantiene la proporción ~37 % de cancelaciones en ambos lados, hallazgo del §2 del EDA).

### ¿Por qué partir *ahora* y no después?

El `test` debe simular **datos que nunca hemos visto**. En producción, cuando llega una reserva nueva, **no sabemos** su desenlace ni el de reservas futuras. Por eso **cualquier número que el preprocesado aprenda** (qué agencias son de riesgo, el vocabulario de columnas *one-hot*, una media de imputación…) debe salir **solo de `train`**.

Si ajustáramos esos parámetros con **todo** el dataset y *luego* partiéramos, información del `test` se **colaría hacia atrás** en el preprocesado: el modelo estaría "haciendo trampa" usando, indirectamente, las respuestas del examen. El resultado es un **ROC-AUC optimista** que **no** se repetirá en producción. La regla es uniforme: **`fit` en `train`, `transform` en ambos** (justo por eso *scikit-learn* separa `.fit()` de `.transform()`).

In [6]:
y = df["is_canceled"]
X = df.drop(columns=["is_canceled"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
)
print("train:", X_train.shape, "| test:", X_test.shape)
print("tasa cancelacion  train: %.3f | test: %.3f" % (y_train.mean(), y_test.mean()))

train: (94850, 29) | test: (23713, 29)
tasa cancelacion  train: 0.373 | test: 0.373


## 4. Reducción de cardinalidad (ajustada **solo en `train`**)

Aplicamos la técnica del **§13 del EDA** a `agent`, `country` y `company`: conservar las categorías con **señal extrema** (cancelan mucho o muy poco) y **soporte suficiente** (`n ≥ 100`), agrupando el resto en `Otros`. La diferencia clave con el EDA: aquí la lista de categorías a conservar (`KEEP`) se calcula **solo con `train`**.

- Umbrales **adaptativos**: tomamos el máximo de tasa de cada variable como referencia; conservamos las de tasa **> 60 %** o **< 30 %** de ese máximo.
- **Nulos**: en `company` los imputamos a una categoría propia **`no_company`** (= *reserva sin empresa*, un estado real, §5); en `agent`/`country` quedan como `Desconocido`.

In [7]:
def key_str(v):
    """IDs como 9.0 -> "9" (claves limpias para one-hot)."""
    if pd.isna(v):
        return None
    if isinstance(v, float) and float(v).is_integer():
        return str(int(v))
    return str(v)


def fit_keep(s_train, y_train, hi_frac=0.60, lo_frac=0.30, min_n=100):
    """Aprende, SOLO con train, qué categorías de `s_train` conservar.

    Conserva las que tienen >= min_n reservas y tasa de cancelación > hi_frac*max
    (alto riesgo) o < lo_frac*max (muy fiables). Devuelve un set de claves (str).
    """
    g = (pd.DataFrame({"v": s_train.values, "y": y_train.values})
           .groupby("v", observed=True)["y"].agg(rate="mean", n="size"))
    enough = g[g["n"] >= min_n]
    m = enough["rate"].max()
    keep = enough[(enough["rate"] > hi_frac * m) | (enough["rate"] < lo_frac * m)].index
    return {key_str(v) for v in keep}


def reduce_apply(s, keep, null_label="Desconocido"):
    """Aplica una lista KEEP: categoría conservada -> su clave; nulo -> null_label;
    resto -> 'Otros'. (Se usa con la `keep` aprendida en train, sobre train y test.)"""
    def f(v):
        if pd.isna(v):
            return null_label
        k = key_str(v)
        return k if k in keep else "Otros"
    return s.map(f)

### 4.1 Ajustar las listas `KEEP` con `train` y aplicarlas a `train` y `test`

In [8]:
REDUCIR = {
    "agent":   {"null_label": "Desconocido"},
    "country": {"null_label": "Desconocido"},
    "company": {"null_label": "no_company"},   # el nulo de company es un estado real
}

keep_lists = {}
for col, opts in REDUCIR.items():
    keep = fit_keep(X_train[col], y_train)          # <-- SOLO train
    keep_lists[col] = keep
    X_train[col] = reduce_apply(X_train[col], keep, opts["null_label"])
    X_test[col] = reduce_apply(X_test[col], keep, opts["null_label"])
    n_orig = X[col].nunique(dropna=True)
    print(f"{col:>8}: {n_orig:>3} categorias -> conservadas {len(keep):>2} (+ Otros + nulos)")

   agent: 333 categorias -> conservadas 50 (+ Otros + nulos)
 country: 177 categorias -> conservadas 14 (+ Otros + nulos)
 company: 347 categorias -> conservadas  8 (+ Otros + nulos)


### 4.2 Prueba de la fuga: la lista `KEEP` cambia si miramos *todo* el dataset

Para que se vea **por qué** ajustamos solo en `train`: si calculáramos la misma lista `KEEP` usando **todo** el dataset (train **+** test), saldría una lista **distinta**. Esa diferencia es, literalmente, información del `test` filtrándose en el preprocesado. Lo comprobamos comparando ambas listas (sin entrenar ningún modelo).

In [9]:
print("Categorías que CAMBIAN si usamos todo el dataset en vez de solo train:\n")
for col in REDUCIR:
    keep_train = keep_lists[col]
    keep_full = fit_keep(X[col], y)                 # <-- mira tambien el test (¡fuga!)
    solo_full = keep_full - keep_train
    solo_train = keep_train - keep_full
    print(f"{col:>8}: |train|={len(keep_train):>2}  |todo|={len(keep_full):>2}  "
          f"-> aparecen {len(solo_full)} y desaparecen {len(solo_train)} categorías")
print("\nCada categoría que cambia es señal del test colándose en el preprocesado.")

Categorías que CAMBIAN si usamos todo el dataset en vez de solo train:

   agent: |train|=50  |todo|=55  -> aparecen 6 y desaparecen 1 categorías
 country: |train|=14  |todo|=14  -> aparecen 1 y desaparecen 1 categorías
 company: |train|= 8  |todo|=10  -> aparecen 2 y desaparecen 0 categorías

Cada categoría que cambia es señal del test colándose en el preprocesado.


## 5. Codificación *one-hot* (vocabulario fijado en `train`)

Codificamos las categóricas con `pd.get_dummies`. Las **numéricas** (incluida `is_repeated_guest`, que ya es 0/1, y las booleanas `has_*`) se quedan tal cual.

El vocabulario de columnas se **fija con `train`**: aplicamos `get_dummies` a `train`, y luego **alineamos** `test` a esas mismas columnas (`reindex`). Así, una categoría que solo aparezca en `test` no crea una columna nueva (no la habríamos visto al entrenar), y una que falte en `test` se rellena con 0. Es otra transformación *fit-on-train*.

In [10]:
CATEGORICAS = [
    "hotel", "arrival_date_month", "meal", "market_segment",
    "distribution_channel", "reserved_room_type",
    "deposit_type", "customer_type", "country", "agent", "company",
]

Xtr = pd.get_dummies(X_train, columns=CATEGORICAS)
Xte = pd.get_dummies(X_test, columns=CATEGORICAS)

# Alinear test al vocabulario de train (fit-on-train del one-hot)
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)

print("Columnas tras one-hot:", Xtr.shape[1])
print("¿Mismas columnas en train y test?", list(Xtr.columns) == list(Xte.columns))

Columnas tras one-hot: 144
¿Mismas columnas en train y test? True


> ✅ Nótese que `assigned_room_type` **ya no aparece**: la quitamos en el §2.1 por **fuga** (se conoce en el *check-in*, §12.8 del EDA). Conservamos `reserved_room_type`, que sí se conoce al reservar.

## 6. Dataset final y guardado

Readjuntamos la etiqueta `is_canceled` y guardamos **dos** ficheros en `data/processed/`: `train.csv` y `test.csv`. Son **dos** dataframes (no uno solo) precisamente porque el preprocesado se ajustó en `train` y se aplicó a `test`: ese es el procedimiento honesto.

In [11]:
train_final = Xtr.copy(); train_final["is_canceled"] = y_train.values
test_final = Xte.copy();  test_final["is_canceled"] = y_test.values

import pathlib
out = pathlib.Path("../../data/processed"); out.mkdir(parents=True, exist_ok=True)
train_final.to_csv(out / "train.csv", index=False)
test_final.to_csv(out / "test.csv", index=False)

print("Guardado en data/processed/:")
print("  train.csv ->", train_final.shape)
print("  test.csv  ->", test_final.shape)
print("\nEjemplo de columnas finales:")
print(list(train_final.columns[:12]), "...")

Guardado en data/processed/:
  train.csv -> (94850, 145)
  test.csv  -> (23713, 145)

Ejemplo de columnas finales:
['lead_time', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes'] ...


## 7. Limitación que **no** arregla el *fit-on-train*: la fuga por duplicación

El *fit-on-train* elimina la fuga de los **parámetros** del preprocesado. Pero queda una fuga **inherente a los datos**: el §9 del EDA mostró que ~34 % de las filas son **duplicados exactos** (reservas en bloque reales). Una partición **aleatoria** reparte copias idénticas entre `train` y `test`, así que el modelo puede **memorizar** una fila en `train` y "reconocer" su gemela en `test`.

Lo cuantificamos: ¿qué porcentaje de filas de `test` tiene un **duplicado exacto** en `train`? (sobre las features ya limpias, antes de codificar).

In [12]:
# Hash por fila sobre las features limpias (mismas columnas en ambos lados)
htr = pd.util.hash_pandas_object(X_train, index=False)
hte = pd.util.hash_pandas_object(X_test, index=False)
share = hte.isin(set(htr)).mean()
print(f"Filas de test con un duplicado exacto en train: {share:.1%}")

Filas de test con un duplicado exacto en train: 34.0%


**Conclusión.** Una fracción nada despreciable del `test` tiene una copia en `train`, lo que **infla** cualquier métrica. Es una **limitación conocida** de este dataset; para una evaluación rigurosa habría que usar una partición **consciente de grupos** (que mantenga los duplicados juntos en un solo lado). Lo dejamos **documentado** (no resuelto), tal como decidimos en el EDA: mantenemos los duplicados porque son reservas reales, pero sabemos que el ROC-AUC final está, probablemente, algo **inflado**.

## 8. Resumen

Hemos convertido el **crudo** en un **dataset listo para modelar** aplicando, una a una, las decisiones del EDA:

| Paso | Qué hicimos | Fuga evitada |
|---|---|---|
| Columnas fuera | `reservation_status(_date)`, `required_car_parking_spaces`, `assigned_room_type`, `arrival_date_year` | Fuga directa y sutil |
| Saneo de filas | fuera reservas sin huéspedes / sin noches / `adr` extremo | Ruido |
| Features | `has_company`, `has_agent`, `noches`; `children` → 0 | — |
| **Partición primero** | `train_test_split` estratificado | Base de todo lo demás |
| Cardinalidad | `agent`/`country`/`company` reducidas, `KEEP` **ajustado en train** | Fuga de parámetros |
| `company` nula | imputada a `no_company` | — |
| One-hot | vocabulario **fijado en train**, `test` alineado por `reindex` | Fuga de vocabulario |
| Guardado | `data/processed/{train,test}.csv` | — |
| Limitación | duplicados → fuga por duplicación **documentada** | (no resuelta) |

> **Lo que nos llevamos:** preparar datos no es solo "limpiar". Es decidir **qué se aprende de los datos** y asegurarse de que eso se aprende **solo del train**. Con `data/processed/{train,test}.csv` ya podemos pasar al **modelado** (`03_modelos_supervisados.ipynb`).